In [1]:
def minimax(depth, node, ismax, values, visited):
    visited.append(node)

    if depth == 3:
        return values[node]

    if ismax:
        left = minimax(depth + 1, node * 2, False, values, visited)
        right = minimax(depth + 1, node * 2 + 1, False, values, visited)
        return max(left, right)
    else:
        left = minimax(depth + 1, node * 2, True, values, visited)
        right = minimax(depth + 1, node * 2 + 1, True, values, visited)
        return min(left, right)


def minimax_limited(depth, node, ismax, values, maxdepth, visited):
    visited.append(node)

    if depth == maxdepth:
        return values[node]

    if ismax:
        left = minimax_limited(depth + 1, node * 2, False, values, maxdepth, visited)
        right = minimax_limited(depth + 1, node * 2 + 1, False, values, maxdepth, visited)
        return max(left, right)
    else:
        left = minimax_limited(depth + 1, node * 2, True, values, maxdepth, visited)
        right = minimax_limited(depth + 1, node * 2 + 1, True, values, maxdepth, visited)
        return min(left, right)


values = [4, 7, 2, 5, 1, 8, 3, 6]

visited_full = []
result_full = minimax(0, 0, True, values, visited_full)

print("minimax: optimal value:", result_full)
print("visited:", visited_full)


values_depth2 = [7, 5, 8, 6]

visited_limited = []
result_limited = minimax_limited(0, 0, True, values_depth2, 2, visited_limited)

print("\ndepth limited optimal value:", result_limited)
print("visited order:", visited_limited)

minimax: optimal value: 6
visited: [0, 0, 0, 0, 1, 1, 2, 3, 1, 2, 4, 5, 3, 6, 7]

depth limited optimal value: 6
visited order: [0, 0, 0, 1, 1, 2, 3]


In [2]:
def alphabeta(depth, node, ismax, values, alpha, beta, visited, pruned):
    visited.append(node)

    if depth == 3:
        return values[node]

    if ismax:
        best = float('-inf')

        val = alphabeta(depth + 1, node * 2, False, values, alpha, beta, visited, pruned)
        best = max(best, val)
        alpha = max(alpha, best)

        if beta <= alpha:
            pruned.append(node * 2 + 1)
            return best

        val = alphabeta(depth + 1, node * 2 + 1, False, values, alpha, beta, visited, pruned)
        best = max(best, val)
        alpha = max(alpha, best)

        return best

    else:
        best = float('inf')

        val = alphabeta(depth + 1, node * 2, True, values, alpha, beta, visited, pruned)
        best = min(best, val)
        beta = min(beta, best)

        if beta <= alpha:
            pruned.append(node * 2 + 1)
            return best

        val = alphabeta(depth + 1, node * 2 + 1, True, values, alpha, beta, visited, pruned)
        best = min(best, val)
        beta = min(beta, best)

        return best


values = [4, 7, 2, 5, 1, 8, 3, 6]

visited = []
pruned = []

result = alphabeta(0, 0, True, values, float('-inf'), float('inf'), visited, pruned)

print("optimal value:", result)
print("visited:", visited)
print("pruned:", pruned)
print("total visited:", len(visited))

optimal value: 6
visited: [0, 0, 0, 0, 1, 1, 2, 3, 1, 2, 4, 5, 3, 6, 7]
pruned: []
total visited: 15


In [3]:
def minimax(depth, node, ismax, values, path, curr):
    curr.append(node)

    if depth == 3 or node >= len(values):
        if node < len(values):
            val = values[node]
        else:
            val = float('-inf')
        return val, curr.copy()

    if ismax:
        best = float('-inf')
        bestpath = []

        for i in range(2):
            val, p = minimax(depth + 1, node * 2 + i, False, values, path, curr.copy())

            if val > best:
                best = val
                bestpath = p

        return best, bestpath

    else:
        best = float('inf')
        bestpath = []

        for i in range(2):
            val, p = minimax(depth + 1, node * 2 + i, True, values, path, curr.copy())

            if val < best:
                best = val
                bestpath = p

        return best, bestpath


def alphabeta(depth, node, ismax, values, alpha, beta, pruned, curr):
    curr.append(node)

    if depth == 3 or node >= len(values):
        if node < len(values):
            val = values[node]
        else:
            val = float('-inf')
        return val, curr.copy()

    if ismax:
        best = float('-inf')
        bestpath = []

        for i in range(2):
            val, p = alphabeta(depth + 1, node * 2 + i, False, values, alpha, beta, pruned, curr.copy())

            if val > best:
                best = val
                bestpath = p

            alpha = max(alpha, best)

            if beta <= alpha:
                pruned.append(node * 2 + (i + 1))
                break

        return best, bestpath

    else:
        best = float('inf')
        bestpath = []

        for i in range(2):
            val, p = alphabeta(depth + 1, node * 2 + i, True, values, alpha, beta, pruned, curr.copy())

            if val < best:
                best = val
                bestpath = p

            beta = min(beta, best)

            if beta <= alpha:
                pruned.append(node * 2 + (i + 1))
                break

        return best, bestpath


values = [4, 7, 2, 9, 1, 8, 3, 6, 10, 12]

mmval, mmpath = minimax(0, 0, True, values, [], [])
print("minimax: optimal value:", mmval)
print("path:", mmpath)

pruned = []

abval, abpath = alphabeta(0, 0, True, values, float('-inf'), float('inf'), pruned, [])
print("\nalpha-beta: optimal value:", abval)
print("path:", abpath)
print("pruned:", pruned)

minimax: optimal value: 7
path: [0, 0, 0, 1]

alpha-beta: optimal value: 7
path: [0, 0, 0, 1]
pruned: [4, 4]


In [1]:
from ortools.sat.python import cp_model

model = cp_model.CpModel()

a = model.new_int_var(0, 3, "a")
b = model.new_int_var(0, 3, "b")
c = model.new_int_var(0, 3, "c")

model.add(a != b)
model.add(b != c)
model.add(a + b <= 4)

solver = cp_model.CpSolver()

status = solver.solve(model)

if status == cp_model.FEASIBLE or status == cp_model.OPTIMAL:
    print("a:", solver.value(a))
    print("b:", solver.value(b))
    print("c:", solver.value(c))
else:
    print("no solution found")

a: 0
b: 1
c: 3


In [2]:
from ortools.sat.python import cp_model

class solverout(cp_model.CpSolverSolutionCallback):

    def __init__(self, a, b, c):
        cp_model.CpSolverSolutionCallback.__init__(self)
        self.a = a
        self.b = b
        self.c = c
        self.count = 0

    def on_solution_callback(self):
        self.count += 1
        print("solution", self.count, "->",
              "a:", self.value(self.a),
              "b:", self.value(self.b),
              "c:", self.value(self.c))


model = cp_model.CpModel()

a = model.new_int_var(0, 3, "a")
b = model.new_int_var(0, 3, "b")
c = model.new_int_var(0, 3, "c")

model.add(a != b)
model.add(b != c)
model.add(a + b <= 4)

solver = cp_model.CpSolver()
solver.parameters.enumerate_all_solutions = True

printer = solverout(a, b, c)

solver.solve(model, printer)

print("\ntotal solutions:", printer.count)

solution 1 -> a: 1 b: 0 c: 1
solution 2 -> a: 0 b: 1 c: 0
solution 3 -> a: 2 b: 1 c: 0
solution 4 -> a: 2 b: 0 c: 1
solution 5 -> a: 1 b: 2 c: 1
solution 6 -> a: 0 b: 2 c: 1
solution 7 -> a: 0 b: 2 c: 0
solution 8 -> a: 1 b: 2 c: 0
solution 9 -> a: 1 b: 2 c: 3
solution 10 -> a: 0 b: 2 c: 3
solution 11 -> a: 0 b: 1 c: 3
solution 12 -> a: 1 b: 0 c: 3
solution 13 -> a: 1 b: 0 c: 2
solution 14 -> a: 0 b: 1 c: 2
solution 15 -> a: 2 b: 1 c: 2
solution 16 -> a: 2 b: 0 c: 2
solution 17 -> a: 2 b: 0 c: 3
solution 18 -> a: 2 b: 1 c: 3
solution 19 -> a: 3 b: 1 c: 3
solution 20 -> a: 3 b: 0 c: 3
solution 21 -> a: 3 b: 0 c: 2
solution 22 -> a: 3 b: 1 c: 2
solution 23 -> a: 3 b: 0 c: 1
solution 24 -> a: 3 b: 1 c: 0
solution 25 -> a: 0 b: 3 c: 0
solution 26 -> a: 0 b: 3 c: 1
solution 27 -> a: 0 b: 3 c: 2
solution 28 -> a: 1 b: 3 c: 2
solution 29 -> a: 1 b: 3 c: 0
solution 30 -> a: 1 b: 3 c: 1

total solutions: 30


In [3]:
from ortools.sat.python import cp_model

model = cp_model.CpModel()

x = model.new_int_var(0, 20, "x")
y = model.new_int_var(0, 20, "y")
z = model.new_int_var(0, 20, "z")

model.add(x + 2 * y + z <= 20)
model.add(3 * x + y <= 18)

model.maximize(4 * x + 2 * y + z)

solver = cp_model.CpSolver()

status = solver.solve(model)

if status == cp_model.OPTIMAL or status == cp_model.FEASIBLE:
    print("optimal value:", solver.objective_value)
    print("x:", solver.value(x))
    print("y:", solver.value(y))
    print("z:", solver.value(z))
else:
    print("no solution")

optimal value: 38.0
x: 6
y: 0
z: 14


In [4]:
from ortools.sat.python import cp_model

model = cp_model.CpModel()

n = 4

q = []
for i in range(n):
    q.append(model.new_int_var(0, n - 1, "q" + str(i)))

model.add_all_different(q)

for i in range(n):
    for j in range(i + 1, n):
        model.add(q[i] - q[j] != i - j)
        model.add(q[i] - q[j] != j - i)

solver = cp_model.CpSolver()

status = solver.solve(model)

if status == cp_model.FEASIBLE or status == cp_model.OPTIMAL:

    board = []

    for i in range(n):
        row = ["_"] * n
        col = solver.value(q[i])
        row[col] = "Q"
        board.append(row)

    for r in board:
        print(" ".join(r))

else:
    print("no solution")

_ _ Q _
Q _ _ _
_ _ _ Q
_ Q _ _


In [5]:
!pip install ortools

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.8/29.8 MB 55.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.4/323.4 kB 26.2 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.6
    Uninstalling protobuf-5.29.6:
      Successfully uninstalled protobuf-5.29.6
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 6.33.6 which is incompatible.
tensorflow 2.19.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.